# Step 5 — BabelBrain: Acoustic & Thermal Simulation

Runs BabelBrain in batch mode (no GUI) using the trajectory file from Step 4 (PlanTUS).
Executes three stages per subject:
- **5a** Domain generation (tissue mask from SimNIBS mesh)
- **5b** Acoustic simulation (FDTD, GPU)
- **5c** Thermal simulation (Bio-heat equation)

**Prerequisites:**
- Step 1: `m2m_{sub_id}/` exists
- Step 4: `*_brainsight.txt` exists (PlanTUS output)

[← Step 4 — PlanTUS](step04_planTUS.ipynb) &nbsp;|&nbsp; **Step 5**


In [ ]:
# Reference card
exec(open(__import__('pathlib').Path(__vsc_ipynb_file__).parent / 'z_references/step5_reference.py').read())

In [ ]:
# -----------------------------------------------------------------------
# Settings — edit here before running
# -----------------------------------------------------------------------
# Hippocampus, offline Draft (200 s worst-case chain).
#
# VTX selects which PlanTUS placement to simulate. Step 4 can leave several
# placements for one target; with VTX set, every output name carries the vertex,
# so two placements can be run and compared side by side instead of
# overwriting each other. Step 5-0 prints the available placements.
#
# ── Site & Thermal profile ───────────────────────────────────────────
SITE_YAML   = "../config/sites/site_UMD_AK.yaml"
STIM_YAML   = "../config/stimulation/stimulation_hippocampus_offline_Pan2025.yaml"
SUB_ID      = "sub-S02"            # then re-check "sub-S01" after its step-4 redo
SUB_LIST_DIR = ""   # subject folder under data_root; "" = the site YAML's sub_list_dir

# ── Target (must match Step 4 output) ────────────────────────────────
TARGET_NAME = "hippocampus_anterior_L_Weizhen_0.05"
TARGET_SIDE = "_L"                  # "_R", "_L", or ""

# ── Which placement ──────────────────────────────────────────────────
# None = the canonical trajectory (whatever step 4c exported last), keeping the
#        original output names so runs made before this setting existed still
#        resolve.
# int  = that vertex's own trajectory, with "_vtx{N}" in every output name.
#        Requires step 4c to have been run for it (it writes the per-vertex
#        trajectory); Step 5-0 lists what is available.
VTX = None

# ── Simulation quality ────────────────────────────────────────────────
PPW         = 6                     # points per wavelength (6=fast, 9=converged)
Z_BEYOND    = 40e-3                 # m (simulation depth beyond target)

# ── CT input ─────────────────────────────────────────────────────────
USE_CT      = False                 # True if CT scan available
CT_PATH     = ""                    # path to CT NIfTI (if USE_CT=True)
CT_TYPE     = 1                     # 1=real CT, 2=ZTE, 3=PETRA

# ── Gel pad ───────────────────────────────────────────────────────────
# Not a setting here. The pad is part of the placement: set once at step 4
# (ADDITIONAL_OFFSET in step04_planTUS.ipynb, --additional-offset in the
# runners), recorded in the depth report, and read from there in Step 5b.
# To change it, re-plan step 4; the distance floor depends on it.

# ── Run options ───────────────────────────────────────────────────────
DRY_RUN     = False                 # True = validate paths only
REUSE_FILES = False                 # True = skip if output already exists

print("Settings loaded.")
print(f"  Subject:              {SUB_ID}")
print(f"  Target:               {TARGET_NAME}{TARGET_SIDE}")
print(f"  Vertex:               {VTX if VTX is not None else 'canonical (last exported)'}")
print(f"  PPW:                  {PPW}")
print(f"  DRY_RUN:              {DRY_RUN}")
print(f"  REUSE_FILES:          {REUSE_FILES}")
print(f"  → Transducer and compute params loaded from config in Step 5-0.")


In [ ]:

# -----------------------------------------------------------------------
# Imports & path setup
# -----------------------------------------------------------------------
import sys
import os
import time
from pathlib import Path
from multiprocessing import Process, Queue

import numpy as np
import yaml as _yaml

NB_DIR = Path(__vsc_ipynb_file__).parent  # notebook directory (cwd-independent anchor)

# ── Make src/ importable ────────────────────────────────────
SRC_DIR = str((NB_DIR / "../src").resolve())
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# Resolve YAMLs relative to notebook location (cwd-independent)
SITE_YAML = str((NB_DIR / SITE_YAML).resolve())
STIM_YAML = str((NB_DIR / STIM_YAML).resolve())

# ── Load site & stim configs ────────────────────────────────────
with open(SITE_YAML) as _f:
    _site = _yaml.safe_load(_f)
with open(STIM_YAML) as _f:
    _stim = _yaml.safe_load(_f)
BASE_ISPPA      = _stim["BaseIsppa"]
THERMAL_PROFILE = _stim["AllDC_PRF_Duration"]
STIM_LABEL      = Path(STIM_YAML).stem.removeprefix("stimulation_")   # in the thermal and figure names

# ── Add BabelBrain to path ──────────────────────────────────────────
_bb_dir = str(Path(os.path.expanduser(_site.get("babelbrain_dir", ""))).resolve())
if _bb_dir and _bb_dir not in sys.path:
    sys.path.insert(0, _bb_dir)

# ── Pipeline utils ────────────────────────────────────────────────
from utils import (
    load_site_config,
    resolve_data_dir,
    normalise_sub_id,
    resolve_sub_dir,
    load_babelbrain_tx_yaml,
    read_trajectory_id_BB,
    fig_dir_for,
    resolve_vtx,
    list_plantus_vertices,
    patch_babelvisco_BB,
    read_depth_report,
    compute_z_steering_BB,
    standoff_check_BB,
    find_roi_mask,
    run_domain_BB,
    run_acoustic_BB,
    run_thermal_BB,
    summarise_acoustic_BB,
    write_tpo_summary_BB,
    plot_acoustic_qc_BB,
    save_acoustic_gui_BB,
    view_acoustic_interactive_BB,
    plot_thermal_qc_BB,
)

print(f"  BabelBrain: {_bb_dir}")
print("Imports complete.")


In [ ]:

# -----------------------------------------------------------------------
# Step 5-0 — Resolve paths and parameters from site config
# -----------------------------------------------------------------------
cfg = load_site_config(SITE_YAML)
if SUB_LIST_DIR:
    cfg["sub_list_dir"] = str(Path(SUB_LIST_DIR).expanduser())
data_dir = resolve_data_dir(cfg)
sub_id_full, sub_id_bare = normalise_sub_id(SUB_ID)

sub_dir = resolve_sub_dir(data_dir, sub_id_bare, sub_id_full)
m2m_dir = sub_dir / f"m2m_{sub_id_full}"
T1W     = str(m2m_dir / "T1.nii.gz")

# ── Transducer parameters ─────────────────────────────────────────────
# To change transducer or device, update the site YAML.
tx_cfg = cfg.get("transducer_cfg", {})
if not tx_cfg:
    raise ValueError("transducer_cfg not found in site config. Check 'transducer:' key in site YAML.")

TX_SYSTEM    = tx_cfg.get("babelbrain_id")
FREQUENCY    = tx_cfg.get("frequency_kHz", 0) * 1e3
if not TX_SYSTEM or not FREQUENCY:
    raise ValueError("babelbrain_id / frequency_kHz missing in the transducer config")

# ── Geometry from BabelBrain's own transducer YAML ────────────────────────────
# The solver's definition of the transducer: aperture, focal length, ring
# diameters, natural focus, steering corrections. The site config only names
# the transducer and the frequency to drive it at; nothing geometric is read
# from it, so the two cannot disagree.
_tx_bb_yaml   = load_babelbrain_tx_yaml(_bb_dir, TX_SYSTEM)
APERTURE      = float(_tx_bb_yaml["TxDiam"])
FOCAL_LENGTH  = float(_tx_bb_yaml["FocalLength"])
IN_DIAMETERS  = _tx_bb_yaml["InDiameters"]
OUT_DIAMETERS = _tx_bb_yaml["OutDiameters"]

# ── Computing device ──────────────────────────────────────────────────
COMPUTING_DEVICE  = cfg.get("computing_device")
COMPUTING_BACKEND = cfg.get("computing_backend")
if COMPUTING_DEVICE is None or COMPUTING_BACKEND is None:
    raise ValueError("computing_device / computing_backend not found in site config.")

# ── Trajectory file from Step 4 ───────────────────────────────────────
plantus_target_folder = (
    m2m_dir / "PlanTUS"
    / f"{sub_id_full}_{TARGET_NAME}_mask_native{TARGET_SIDE}"
)
_label = f"{sub_id_full}_T1w_{TARGET_NAME}{TARGET_SIDE}"

# Show what step 4 left behind. inter = 0.00 means the beam axis misses the ROI.
_placements = list_plantus_vertices(plantus_target_folder)

# Which placement to use. One rule, the same one step 4c follows:
#   VTX = None -> the newest vtx* folder;  VTX = N -> vertex N.
# That vertex then supplies the trajectory AND the depth report, so the two
# cannot disagree — reading the depth report of a different placement is how
# sub-S01 once got an 11 mm steering error.
VTX_EFF = resolve_vtx(plantus_target_folder, VTX)

trajectory_file = str(plantus_target_folder
                      / f"{_label}_vtx{VTX_EFF}_target_coordinates_Brainsight_PlanTUS.txt")
if not os.path.isfile(trajectory_file):
    raise FileNotFoundError(
        f"No trajectory for vtx{VTX_EFF}.\n"
        f"  expected: {os.path.basename(trajectory_file)}\n\n"
        f"  Per-vertex trajectories were added on 2026-08-04 (commit 1ca2bbe).\n"
        f"  A placement exported before that has only the canonical file, which\n"
        f"  carries no vertex identity — so step 4c has to be re-run for it.\n\n"
        f"  Fix — in step04_planTUS.ipynb, restart the kernel, then set:\n"
        f'      SUB_ID          = "{SUB_ID}"\n'
        f'      TARGET_NAME     = "{TARGET_NAME}"\n'
        f'      TARGET_SIDE     = "{TARGET_SIDE}"\n'
        f"      VTX             = {VTX_EFF}\n"
        f"      REUSE_PLACEMENT = True      # 4b skips; no wb_view needed\n"
        f"  Run cells 1-4 and 4c, then re-run this cell."
    )

print(f"\nSubject:      {sub_id_full}")
print(f"m2m_dir:      {m2m_dir}")
print(f"Vertex:       {VTX if VTX is not None else 'canonical (last exported)'}")
print(f"Trajectory:   {trajectory_file}")
print(f"Transducer:   {TX_SYSTEM}  {int(FREQUENCY/1e3)} kHz  ∅{APERTURE*1e3:.1f}mm  f={FOCAL_LENGTH*1e3:.1f}mm")
print(f"Device:       {COMPUTING_DEVICE} (backend={COMPUTING_BACKEND})")
print(f"Use CT:       {USE_CT}")


In [ ]:

# -----------------------------------------------------------------------
# Step 5a — Domain generation
# Generates tissue domain from SimNIBS mesh + trajectory.
# Output: m2m_dir/{prefix}BabelViscoInput.nii.gz
#
# With VTX set, the vertex goes into ID, and prefix / _field_target both derive
# from ID — so domain, acoustic, thermal, Summary.csv and every figure become
# per-vertex automatically and two placements no longer overwrite each other.
# VTX=None keeps the original names so earlier runs still resolve.
# -----------------------------------------------------------------------
ID          = read_trajectory_id_BB(trajectory_file)   # matches BabelBrain GUI (Target name from file content)
if VTX is not None:
    ID     += f"_vtx{VTX}"
prefix      = f"{ID}_{TX_SYSTEM}_{int(FREQUENCY/1e3)}kHz_{PPW}PPW_"
domain_file = str(m2m_dir / f"{prefix}BabelViscoInput.nii.gz")
print(f"ID:     {ID}")
print(f"prefix: {prefix}")

domain_file = run_domain_BB(
    m2m_dir, T1W, trajectory_file, prefix,
    COMPUTING_BACKEND, COMPUTING_DEVICE,
    FREQUENCY, PPW, domain_file,
    use_ct=USE_CT, ct_path=CT_PATH, ct_type=CT_TYPE,
    reuse_files=REUSE_FILES, dry_run=DRY_RUN,
)


In [ ]:

# -----------------------------------------------------------------------
# Step 5b — ZSteering calculation (verify before running simulation)
# Reads the PlanTUS depth report and computes the electronic steering
# offset required to shift the focus to the target depth.
#
# vtx=VTX_EFF makes the depth report match the trajectory. When several placements
# exist and VTX is None, read_depth_report raises instead of guessing: it used
# to take the alphabetically first report, which for sub-S01 meant steering to
# 114.1 mm while the trajectory was 103.0 mm — an 11 mm error, silently.
#
# The gel pad is read from the same report (planned at step 4); it is not set
# in this notebook.
# -----------------------------------------------------------------------
Z_STEERING, TX_MECH_ADJ_Z = compute_z_steering_BB(
    plantus_target_folder,
    tx_cfg,
    bb_tx_yaml=_tx_bb_yaml,
    vtx=VTX_EFF,
)


In [ ]:

# -----------------------------------------------------------------------
# Step 5b — Run acoustic simulation
# Output: *DataForSim.h5
# -----------------------------------------------------------------------
_field_target = f"{ID}_{TX_SYSTEM}"

acoustic_file = run_acoustic_BB(
    m2m_dir, _field_target, TX_SYSTEM,
    FREQUENCY, APERTURE, FOCAL_LENGTH,
    IN_DIAMETERS, OUT_DIAMETERS,
    COMPUTING_BACKEND, COMPUTING_DEVICE, PPW,
    z_steering=Z_STEERING,
    tx_mech_adj_z=TX_MECH_ADJ_Z,
    z_beyond=Z_BEYOND, use_ct=USE_CT,
    reuse_files=REUSE_FILES, dry_run=DRY_RUN,
)

# The standoff the field was computed with must be the pad the placement plans
# (step 4 report). Raises beyond 5 mm; a pad counted twice read 20 mm here and
# was invisible in every focus metric (2026-09-20).
STANDOFF = (standoff_check_BB(acoustic_file, plantus_target_folder, VTX_EFF, _tx_bb_yaml)
            if acoustic_file else None)

In [ ]:

# -----------------------------------------------------------------------
# Step 5b — Summary & QC figure
#
# summarise_acoustic_BB  → prints key metrics to console + saves HTML table
#                           Colour-coded safety flags (ITRUSST 2023):
#                           green = OK | orange = borderline | red = concern
#
# plot_acoustic_qc_BB    → saves PNG + displays it here:
#                           2 panels (sagittal / coronal through target)
#                           colour scale = p² / p²_max  (proportional to ISPPA)
#                           grey contours = skull, brown = skin surface,
#                           cyan contour  = target ROI (from Step 3)
#                           white cross = focal target
#
# view_acoustic_interactive_BB → interactive 3-D viewers (drag / zoom / slice):
#   [1] p / p_max overlaid on native T1   (0 = no pressure, 1 = peak)
#   [2] target ROI outline on T1          (from Step 3 native mask)
#   Both saved as HTML to fig_dir.
# -----------------------------------------------------------------------
# figures/{target}/ for acoustic, figures/{target}/{stim}/ for thermal.
# Acoustic sits above the protocol because it does not depend on it — the same
# acoustic run feeds every thermal protocol. fig_dir_for() is the single place
# this layout is defined, shared with run_babelbrain.py and the comparison PDF.
fig_dir = fig_dir_for(sub_dir, TARGET_NAME)

_roi_nii = find_roi_mask(plantus_target_folder)
if not _roi_nii:
    print(f'[QC] No native ROI mask found in: {plantus_target_folder}')

summarise_acoustic_BB(acoustic_file, fig_dir=fig_dir, standoff=STANDOFF)

# The BabelBrain GUI's own Step-2 plot, ported line for line. Use this one to
# compare against the GUI — it is the same figure.
save_acoustic_gui_BB(acoustic_file, fig_dir=fig_dir)

# Anatomical QC. Normalised to the WHOLE-DOMAIN maximum, which for a deep
# target sits in cortical bone, so the brain field is squeezed below 0.2 and
# the whole beam renders blue. That is not a fault — it is what "the skull
# takes more pressure than the brain" looks like. The GUI figure above zeroes
# everything outside brain and renormalises, which is why its beam spans the
# full colour scale. Read this one for the ROI contour and the anatomy.
plot_acoustic_qc_BB(acoustic_file, fig_dir, ID, TX_SYSTEM, FREQUENCY, PPW, roi_nii=_roi_nii)
view_acoustic_interactive_BB(acoustic_file, T1W, roi_nii=_roi_nii, fig_dir=fig_dir)


In [ ]:
# -----------------------------------------------------------------------
# Step 5c — Thermal simulation
# Solves Bio-heat Thermal Equation (BHTE) for all DC/PRF/Duration combos.
# Input: acoustic_file (skull h5)   Output: *_{STIM_LABEL}_AllCombinations.h5
#
# patch_babelvisco_BB() corrects the BabelViscoFDTD `intparams` dtype bug
# (uint32 vs the shader's `constant int *`), which raised OverflowError on
# LocationMonitoring=-1 under numpy>=2. The fix is written to the installed
# file on disk, because BHTE runs in a nested spawned process that would not
# inherit an in-memory patch. Safe to call every run; idempotent.
#
# Backend comes from the site YAML (computing_backend). Metal (3) is the only
# backend verified on this M1 Mac. Do not use OpenCL (deprecated on macOS) or
# MLX (untested dev build), and never pass an unrecognised backend string —
# BHTE silently returns zeroed temperature/dose arrays in that case.
# -----------------------------------------------------------------------
patch_babelvisco_BB()

allcomb_h5 = run_thermal_BB(
    acoustic_file,
    THERMAL_PROFILE, BASE_ISPPA, FREQUENCY, TX_SYSTEM,
    COMPUTING_BACKEND,   # from site YAML: 3 = Metal
    COMPUTING_DEVICE,
    STIM_LABEL,
    reuse_files=REUSE_FILES, dry_run=DRY_RUN,
)


In [ ]:
# -----------------------------------------------------------------------
# Step 5c — TPO setting (free-field ISPPA back-calculation)
#
# BaseIsppa in the stimulation YAML is the *in-situ* ISPPA at the focal point.
# The NeuroFUS TPO, however, takes *free-field (in-water)* ISPPA as its input
# and computes the per-channel power itself. The two differ by the skull/scalp
# insertion loss, which the simulation gives us:
#
#     required free-field ISPPA = BaseIsppa / derating_ratio
#     derating_ratio = (max |p| in brain / max |p| in the water run) ** 2
#
# Writes *_Summary.csv beside the thermal output, one row per combination.
# When the transducer YAML has calibration.isppa_w_per_cm2 (CTX-500 does, the
# UMD DPX-500 does not) the requirement is also compared with the calibrated
# reference. Exceeding it is reported, not blocked — the TPO power limit is
# adjustable, though the vendor calls operating above it off-label.
# -----------------------------------------------------------------------
summary_csv = write_tpo_summary_BB(
    acoustic_file, allcomb_h5, tx_cfg,
    sub_id_full, TARGET_NAME, TARGET_SIDE,
)


In [ ]:

# -----------------------------------------------------------------------
# Step 5-QC — Thermal visualisation
#
# Figures produced (saved to {sub_dir}/figures/{TARGET_NAME}/{stim}/):
#   Thermal  : ΔT maps (end-FUS + after cool-down), temporal profiles,
#              safety summary (ΔT + CEM43) — one figure per combo
#
# Modelled on BabelBrain GUI output (Simulation-12, -15).
# -----------------------------------------------------------------------
plot_thermal_qc_BB(
    allcomb_h5, _field_target, m2m_dir,
    fig_dir_for(sub_dir, TARGET_NAME, STIM_LABEL), ID,
    sub_id_full, TARGET_NAME, TARGET_SIDE, TX_SYSTEM, FREQUENCY,
    acoustic_file=acoustic_file,
    stim_label=STIM_LABEL,
)
